# Notebook 03 — Generate RAG Documents (Google Colab)

This notebook reads the DuckDB tables created by Notebook 02 and generates text documents for retrieval-augmented generation (RAG).

Outputs are saved under:
`MyDrive/RAG UFC-AI-Analyst/documents/`

It creates:
- `fighters/` — one profile per fighter
- `fights/` — one summary per fight
- `events/` — one summary per event

The notebook does not call an LLM or create embeddings. Notebook 04 builds the vector store from these documents.

**Data-quality note:** `fighter_a` / `fighter_b` winner labels depend on the source's red/blue mapping. The document generator includes the outcome as recorded in the source and only states a winner when the derived performance table contains exactly one `is_winner=True` row for that fight.


In [1]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
# Cell 2 — Install dependencies
%pip install -q duckdb pandas pyarrow tqdm


In [3]:
# Cell 3 — Configure project paths
from pathlib import Path
import re
import json
import pandas as pd
import duckdb
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/RAG UFC-AI-Analyst")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
DB_PATH = DATA_DIR / "ufc.duckdb"
DOCS_DIR = PROJECT_ROOT / "documents"
FIGHTERS_DIR = DOCS_DIR / "fighters"
FIGHTS_DIR = DOCS_DIR / "fights"
EVENTS_DIR = DOCS_DIR / "events"

for folder in [DOCS_DIR, FIGHTERS_DIR, FIGHTS_DIR, EVENTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

if not DB_PATH.is_file():
    raise FileNotFoundError(
        f"DuckDB database not found: {DB_PATH}. Run Notebook 02 first."
    )

print("Project:", PROJECT_ROOT)
print("Database:", DB_PATH)
print("Documents output:", DOCS_DIR)


Project: /content/drive/MyDrive/RAG UFC-AI-Analyst
Database: /content/drive/MyDrive/RAG UFC-AI-Analyst/data/ufc.duckdb
Documents output: /content/drive/MyDrive/RAG UFC-AI-Analyst/documents


In [4]:
# Cell 4 — Connect and inspect tables
conn = duckdb.connect(str(DB_PATH), read_only=True)
available_tables = conn.execute("SHOW TABLES").fetchdf()["name"].tolist()
print("Database tables:", available_tables)

required_tables = {"events", "fighters", "fights", "fight_performances"}
missing_tables = required_tables - set(available_tables)
if missing_tables:
    conn.close()
    raise ValueError(f"Missing tables in DuckDB: {sorted(missing_tables)}")

for table_name in sorted(required_tables):
    count = conn.execute(f'SELECT COUNT(*) FROM "{table_name}"').fetchone()[0]
    print(f"{table_name}: {count:,} rows")


Database tables: ['events', 'fight_performances', 'fighters', 'fights']
events: 794 rows
fight_performances: 17,784 rows
fighters: 2,747 rows
fights: 8,892 rows


In [5]:
# Cell 5 — Load the tables into pandas
events = conn.execute("SELECT * FROM events").fetchdf()
fighters = conn.execute("SELECT * FROM fighters").fetchdf()
fights = conn.execute("SELECT * FROM fights").fetchdf()
performances = conn.execute("SELECT * FROM fight_performances").fetchdf()

# Read source outcome and participant names from the cleaned CSV for reliable context.
CLEAN_CSV_PATH = PROCESSED_DIR / "ufc_fights_clean.csv"
if CLEAN_CSV_PATH.is_file():
    source = pd.read_csv(CLEAN_CSV_PATH, low_memory=False)
else:
    source = None
    print("Warning: cleaned CSV not found; some fight/event document context may be limited.")

print("Loaded:", {
    "events": events.shape,
    "fighters": fighters.shape,
    "fights": fights.shape,
    "performances": performances.shape,
})


Loaded: {'events': (794, 4), 'fighters': (2747, 5), 'fights': (8892, 11), 'performances': (17784, 129)}


In [6]:
# Cell 6 — Utility functions for safe text and filenames
def clean_text(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if text.lower() in {"nan", "nat", "none", "<na>"}:
        return ""
    return re.sub(r"\s+", " ", text)

def safe_filename(value, fallback="unknown"):
    text = clean_text(value)
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text).strip("._-")
    return (text[:100] or fallback)

def fmt(value, suffix=""):
    text = clean_text(value)
    return f"{text}{suffix}" if text else "Unknown"


In [7]:
# Cell 7 — Normalize date and ID keys
for frame, col in [(events, "event_date")]:
    if col in frame.columns:
        frame[col] = pd.to_datetime(frame[col], errors="coerce")

# String keys prevent joins from failing when IDs are read with different dtypes.
for frame, cols in [
    (events, ["event_id"]),
    (fighters, ["fighter_id"]),
    (fights, ["fight_id", "event_id"]),
    (performances, ["fight_id", "fighter_id", "opponent_id"]),
]:
    for col in cols:
        if col in frame.columns:
            frame[col] = frame[col].astype("string")

if source is not None:
    for col in ["fight_id", "event_id", "fighter_a_id", "fighter_b_id"]:
        if col in source.columns:
            source[col] = source[col].astype("string")
    if "event_date" in source.columns:
        source["event_date"] = pd.to_datetime(source["event_date"], errors="coerce")

fighter_by_id = (
    fighters.dropna(subset=["fighter_id"])
    .drop_duplicates("fighter_id")
    .set_index("fighter_id")
    .to_dict(orient="index")
)
event_by_id = (
    events.dropna(subset=["event_id"])
    .drop_duplicates("event_id")
    .set_index("event_id")
    .to_dict(orient="index")
)


In [8]:
# Cell 8 — Build fight-level context
# Include source columns where available; keep IDs as strings for consistent matching.
if source is not None:
    source_fight_cols = [
        c for c in [
            "fight_id", "event_id", "event_name", "event_date", "event_location",
            "fighter_a_id", "fighter_a_name", "fighter_b_id", "fighter_b_name",
            "fight_outcome", "weight_class", "method", "round", "time",
            "time_format", "referee", "details"
        ] if c in source.columns
    ]
    source_fights = source[source_fight_cols].drop_duplicates("fight_id").copy()
else:
    source_fights = fights.copy()

# Attach a single winner only when exactly one performance row is marked True.
winner_rows = performances.loc[
    performances["is_winner"].astype("boolean").fillna(False)
].copy()
winner_counts = winner_rows.groupby("fight_id").size()
unique_winner_fights = set(winner_counts[winner_counts == 1].index.astype(str))
winner_id_map = (
    winner_rows[winner_rows["fight_id"].astype(str).isin(unique_winner_fights)]
    .drop_duplicates("fight_id")
    .set_index("fight_id")["fighter_id"]
    .astype(str)
    .to_dict()
)

performance_by_fight = {
    str(fid): group.copy()
    for fid, group in performances.groupby(performances["fight_id"].astype(str), sort=False)
}

source_fight_by_id = (
    source_fights.dropna(subset=["fight_id"])
    .drop_duplicates("fight_id")
    .set_index("fight_id")
    .to_dict(orient="index")
)
print("Fight records available for document generation:", len(source_fight_by_id))
print("Fights with a unique derived winner:", len(winner_id_map))


Fight records available for document generation: 8892
Fights with a unique derived winner: 8891


In [9]:
# Cell 9 — Generate fighter profile documents
# Each fighter document includes known attributes and a compact fight-history summary.
fighter_documents = 0
fighter_ids_in_performances = set(performances["fighter_id"].dropna().astype(str))

for _, fighter in tqdm(fighters.iterrows(), total=len(fighters), desc="Fighter documents"):
    fighter_id = clean_text(fighter.get("fighter_id"))
    if not fighter_id:
        continue

    name = fmt(fighter.get("name"))
    dob = fmt(fighter.get("dob"))
    height = fmt(fighter.get("height"))
    reach = fmt(fighter.get("reach"))

    history = performances[
        performances["fighter_id"].astype("string") == fighter_id
    ].copy()

    total_fights = len(history)
    wins = int(history["is_winner"].fillna(False).sum()) if "is_winner" in history else 0
    known_results = int(history["is_winner"].notna().sum()) if "is_winner" in history else 0
    losses = max(known_results - wins, 0)
    unknown_results = total_fights - known_results

    lines = [
        f"# UFC Fighter Profile: {name}",
        f"Fighter ID: {fighter_id}",
        f"Date of birth: {dob}",
        f"Height: {height}",
        f"Reach: {reach}",
        "",
        "## Career summary",
        f"Recorded fights in this dataset: {total_fights}",
        f"Wins based on derived source labels: {wins}",
        f"Losses based on derived source labels: {losses}",
        f"Fights with unknown result: {unknown_results}",
        "",
        "## Recent recorded fights",
    ]

    if history.empty:
        lines.append("No fight performance records found.")
    else:
        recent = history.head(15)
        for _, perf in recent.iterrows():
            fight_id = clean_text(perf.get("fight_id"))
            fight_context = source_fight_by_id.get(fight_id, {})
            opponent_id = clean_text(perf.get("opponent_id"))
            opponent = fighter_by_id.get(opponent_id, {})
            opponent_name = clean_text(opponent.get("name")) or "Unknown opponent"
            outcome_value = perf.get("is_winner")
            if pd.isna(outcome_value):
                result = "unknown result"
            else:
                result = "win" if bool(outcome_value) else "loss"
            event_name = clean_text(fight_context.get("event_name")) or "Unknown event"
            event_date = clean_text(fight_context.get("event_date")) or "Unknown date"
            weight_class = clean_text(fight_context.get("weight_class")) or "Unknown weight class"
            lines.append(
                f"- Fight ID {fight_id}: {result} vs {opponent_name}; "
                f"event: {event_name}; date: {event_date}; weight class: {weight_class}."
            )

    filename = FIGHTERS_DIR / f"{safe_filename(name)}_{safe_filename(fighter_id)}.txt"
    filename.write_text("\n".join(lines) + "\n", encoding="utf-8")
    fighter_documents += 1

print("Fighter documents generated:", fighter_documents)


Fighter documents:   0%|          | 0/2747 [00:00<?, ?it/s]

Fighter documents generated: 2747


In [10]:
# Cell 10 — Generate fight summary documents
fight_documents = 0

for fight_id, record in tqdm(source_fight_by_id.items(), total=len(source_fight_by_id), desc="Fight documents"):
    perf_group = performance_by_fight.get(str(fight_id), pd.DataFrame())
    event_id = clean_text(record.get("event_id"))
    event = event_by_id.get(event_id, {})

    event_name = clean_text(record.get("event_name")) or clean_text(event.get("event_name")) or "Unknown event"
    event_date_value = record.get("event_date", event.get("event_date", ""))
    event_date = clean_text(event_date_value) or "Unknown date"
    location = clean_text(record.get("event_location")) or clean_text(event.get("event_location")) or "Unknown location"
    fighter_a_name = clean_text(record.get("fighter_a_name")) or "Unknown fighter A"
    fighter_b_name = clean_text(record.get("fighter_b_name")) or "Unknown fighter B"
    outcome = clean_text(record.get("fight_outcome")) or "Unknown"
    weight_class = clean_text(record.get("weight_class")) or "Unknown"
    method = clean_text(record.get("method")) or "Unknown"
    round_value = clean_text(record.get("round")) or "Unknown"
    time_value = clean_text(record.get("time")) or "Unknown"
    time_format = clean_text(record.get("time_format")) or "Unknown"
    referee = clean_text(record.get("referee")) or "Unknown"
    details = clean_text(record.get("details")) or "None recorded"

    winner_id = winner_id_map.get(str(fight_id))
    winner_name = clean_text(fighter_by_id.get(winner_id, {}).get("name")) if winner_id else ""
    winner_line = f"Derived winner: {winner_name} (fighter ID {winner_id})" if winner_name else "Derived winner: Unknown / not uniquely determined"

    lines = [
        f"# UFC Fight Summary: {fighter_a_name} vs {fighter_b_name}",
        f"Fight ID: {fight_id}",
        f"Event ID: {event_id or 'Unknown'}",
        f"Event: {event_name}",
        f"Date: {event_date}",
        f"Location: {location}",
        f"Fighter A: {fighter_a_name}",
        f"Fighter B: {fighter_b_name}",
        f"Source outcome label: {outcome}",
        winner_line,
        f"Weight class: {weight_class}",
        f"Method: {method}",
        f"Ending round: {round_value}",
        f"Time: {time_value}",
        f"Time format: {time_format}",
        f"Referee: {referee}",
        f"Details: {details}",
        "",
        "## Fighter performance statistics",
    ]

    if perf_group.empty:
        lines.append("No performance statistics available.")
    else:
        for _, perf in perf_group.iterrows():
            fighter_id = clean_text(perf.get("fighter_id"))
            fighter_name = clean_text(fighter_by_id.get(fighter_id, {}).get("name")) or fighter_id or "Unknown fighter"
            lines.append(f"### {fighter_name} (fighter ID {fighter_id})")
            metric_count = 0
            for col, value in perf.items():
                if col in {"fight_id", "fighter_id", "opponent_id", "is_winner"}:
                    continue
                text_value = clean_text(value)
                if text_value:
                    lines.append(f"- {col.replace('_', ' ')}: {text_value}")
                    metric_count += 1
                if metric_count >= 50:
                    lines.append("- Additional metrics omitted from this text document.")
                    break

    filename = FIGHTS_DIR / f"fight_{safe_filename(fight_id)}.txt"
    filename.write_text("\n".join(lines) + "\n", encoding="utf-8")
    fight_documents += 1

print("Fight documents generated:", fight_documents)


Fight documents:   0%|          | 0/8892 [00:00<?, ?it/s]

Fight documents generated: 8892


In [11]:
# Cell 11 — Generate event summary documents
event_documents = 0

if source is not None:
    event_source = source.copy()
else:
    event_source = fights.merge(events, on="event_id", how="left", suffixes=("", "_event"))

for event_id, group in tqdm(event_source.groupby("event_id", dropna=False), desc="Event documents"):
    event_id_text = clean_text(event_id)
    if not event_id_text:
        continue

    first = group.iloc[0]
    event_name = clean_text(first.get("event_name")) or clean_text(event_by_id.get(event_id_text, {}).get("event_name")) or "Unknown event"
    event_date = clean_text(first.get("event_date")) or "Unknown date"
    location = clean_text(first.get("event_location")) or "Unknown location"

    lines = [
        f"# UFC Event Summary: {event_name}",
        f"Event ID: {event_id_text}",
        f"Event date: {event_date}",
        f"Location: {location}",
        f"Recorded fights: {len(group)}",
        "",
        "## Fight card",
    ]

    for _, fight in group.iterrows():
        a_name = clean_text(fight.get("fighter_a_name")) or "Unknown fighter A"
        b_name = clean_text(fight.get("fighter_b_name")) or "Unknown fighter B"
        outcome = clean_text(fight.get("fight_outcome")) or "Unknown outcome"
        weight_class = clean_text(fight.get("weight_class")) or "Unknown weight class"
        method = clean_text(fight.get("method")) or "Unknown method"
        lines.append(f"- {a_name} vs {b_name}; outcome: {outcome}; weight class: {weight_class}; method: {method}.")

    filename = EVENTS_DIR / f"{safe_filename(event_name)}_{safe_filename(event_id_text)}.txt"
    filename.write_text("\n".join(lines) + "\n", encoding="utf-8")
    event_documents += 1

print("Event documents generated:", event_documents)


Event documents:   0%|          | 0/794 [00:00<?, ?it/s]

Event documents generated: 794


In [12]:
# Cell 12 — Write a manifest and verify generated files
manifest = {
    "project_root": str(PROJECT_ROOT),
    "database": str(DB_PATH),
    "source_csv": str(CLEAN_CSV_PATH) if CLEAN_CSV_PATH.exists() else None,
    "documents": {
        "fighters": fighter_documents,
        "fights": fight_documents,
        "events": event_documents,
    },
    "directories": {
        "fighters": str(FIGHTERS_DIR),
        "fights": str(FIGHTS_DIR),
        "events": str(EVENTS_DIR),
    },
}
manifest_path = DOCS_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

for label, folder in [("fighters", FIGHTERS_DIR), ("fights", FIGHTS_DIR), ("events", EVENTS_DIR)]:
    files = list(folder.glob("*.txt"))
    print(f"{label}: {len(files)} .txt files")
    if files:
        print("  Example:", files[0].name)
        print("  First 8 lines:")
        print("\n".join(files[0].read_text(encoding="utf-8").splitlines()[:8]))

print("Manifest saved:", manifest_path)


fighters: 2747 .txt files
  Example: ROYCE_GRACIE_429e7d3725852ce9.txt
  First 8 lines:
# UFC Fighter Profile: ROYCE GRACIE
Fighter ID: 429e7d3725852ce9
Date of birth: 1966-12-12 00:00:00
Height: 73.0
Reach: Unknown

## Career summary
Recorded fights in this dataset: 9
fights: 8892 .txt files
  Example: fight_00835554f95fa911.txt
  First 8 lines:
# UFC Fight Summary: ROYCE GRACIE vs PATRICK SMITH
Fight ID: 00835554f95fa911
Event ID: ufc-2-no-way-out
Event: UFC 2: No Way Out
Date: 1994-03-11 00:00:00
Location: Denver, Colorado, USA
Fighter A: ROYCE GRACIE
Fighter B: PATRICK SMITH
events: 794 .txt files
  Example: Noche_UFC_Silva_vs._Delgado_noche-ufc-silva-vs-delgado.txt
  First 8 lines:
# UFC Event Summary: Noche UFC: Silva vs. Delgado
Event ID: noche-ufc-silva-vs-delgado
Event date: 2026-09-12 00:00:00
Location: Glendale, Arizona, USA
Recorded fights: 13

## Fight card
- Jean Silva vs Jose Delgado; outcome: red_win; weight class: Featherweight; method: Submission.
Manifest saved: /con

In [13]:
# Cell 13 — Close DuckDB connection
conn.close()
print("Notebook 03 completed. Documents are saved in Google Drive.")


Notebook 03 completed. Documents are saved in Google Drive.
